# Embedding

## Reading and Preprocessing data

In [1]:
MED_DATA_PATH = "./TP_ISD2020/QUAERO_FrenchMed/QUAERO_FrenchMed_traindev.ospl"
PRESS_DATA_PATH = "./TP_ISD2020/QUAERO_FrenchPress/QUAERO_FrenchPress_traindev.ospl"

In [2]:
# Load the data file 
def load_data(Path):
    with open(Path) as f:
        data = f.readlines()
    return data

# Preprocess the text by tokenizing it into the input format of the models
def preprocess_data(data):
    tokenized_data = [line.lower().strip().split() for line in data]
    return tokenized_data

In [3]:
# Load and preprocess the data

med_data = load_data(MED_DATA_PATH)
press_data = load_data(PRESS_DATA_PATH)

tokenized_med_data = preprocess_data(med_data)
tokenized_press_data = preprocess_data(press_data)

In [4]:
# Preview of the results for medical data
tokenized_med_data[0:5]  # Display the first 5 tokenized sentences from the medical data

[['emea', '/', 'h', '/', 'c', '/', '551'],
 ['prialt'],
 ['qu', '’', 'est', 'ce', 'que', 'prialt', '?'],
 ['prialt',
  'est',
  'une',
  'solution',
  'pour',
  'perfusion',
  'contenant',
  'le',
  'principe',
  'actif',
  'ziconotide',
  ',',
  'à',
  'des',
  'concentrations',
  'de',
  '100',
  'ou',
  '25',
  'microgrammes',
  'par',
  'millilitre',
  '.'],
 ['dans', 'quel', 'cas', 'prialt', 'est', '-', 'il', 'utilisé', '?']]

In [5]:
# Preview of the results for press data
tokenized_press_data[0:5]  # Display the first 5 tokenized sentences from the medical data

[['patricia',
  'martin',
  ',',
  'que',
  'voici',
  ',',
  'que',
  'voilà',
  '!',
  'oh',
  ',',
  'bonjour',
  'nicolas',
  'stoufflet',
  '.'],
 ['france', 'inter', ',', '7', 'heures', '.'],
 ['le', 'journal', ',', 'simon', 'tivolle', '.'],
 ['bonjour', '!', 'lundi', '7', 'décembre', '.'],
 ['deux',
  'incendies',
  'cette',
  'nuit',
  'en',
  'région',
  'parisienne',
  ',',
  'dans',
  'une',
  'maison',
  'de',
  'retraite',
  'de',
  'livry-gargan',
  'en',
  'seine-saint-denis',
  ',',
  '7',
  'personnes',
  'ont',
  'péri',
  'dans',
  'les',
  'flammes',
  '.']]

## Word Embeddings Training 

### Word2Vec Models

In [6]:
from gensim.models import Word2Vec

# Helper function to train Word2Vec models with specified parameters
def w2v(sentences, sg, vector_size=100, window=5):

	model = Word2Vec(
		sentences=sentences,
		vector_size=vector_size,
		window=window,
		min_count=1,
		sg=sg,
		seed=42,
		workers=1
	)
	return model

# CBOW model for medical data
w2v_cbow_med_model = w2v(tokenized_med_data, sg=0)

# Skipgram model for medical data
w2v_skipgram_med_model = w2v(tokenized_med_data, sg=1)

# CBOW model for press data
w2v_cbow_press_model = w2v(tokenized_press_data, sg=0)

# Skipgram model for press data
w2v_skipgram_press_model = w2v(tokenized_press_data, sg=1)


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


### Fast Text models

In [7]:
from gensim.models import FastText

# Helper function to create a FastText model with the specified parameters
def fasttext_cbow(sentences, vector_size=100, window=5, sg=0):
    model = FastText(
		sentences=sentences,
		vector_size=vector_size,
		window=window,
		min_count=1,
		sg=sg,
		seed=42,
		workers=1
	)
    return model

# CBOW model for medical data using FastText
ft_cbow_med_model = fasttext_cbow(tokenized_med_data)

# CBOW model for press data using FastText
ft_cbow_press_model = fasttext_cbow(tokenized_press_data)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


### Saving the embeddings

In [8]:
import os

os.makedirs("embeddings", exist_ok=True)

all_models = {
    "w2v_cbow_med": w2v_cbow_med_model,
    "w2v_skipgram_med": w2v_skipgram_med_model,
    "ft_cbow_med": ft_cbow_med_model,
    "w2v_cbow_press": w2v_cbow_press_model,
    "w2v_skipgram_press": w2v_skipgram_press_model,
    "ft_cbow_press": ft_cbow_press_model,
}

for name, model in all_models.items():
    model.wv.save_word2vec_format(f"embeddings/{name}.bin", binary=True)

### Sanity check

In [9]:
# Sanity check function to verify the models and their word vectors

med_models = [w2v_cbow_med_model, w2v_skipgram_med_model, ft_cbow_med_model]
press_models = [w2v_cbow_press_model, w2v_skipgram_press_model, ft_cbow_press_model]

def sanityquick_check(models, words):
	for model in models:
		print(f"Model: {model}")
		for word in words:
			if word in model.wv:
				print(f"Word: {word}, Vector: {model.wv[word][:5]}, Vector Length: {len(model.wv[word])}")
			else:
				print(f"Word: {word} not found in the model.")
		print("\n")
  

In [10]:
# Sanity check for medical models with a list of relevant words
sanityquick_check(med_models, ["cancer", "santé", "maladie", "traitement", "virus"])

Model: Word2Vec<vocab=8088, vector_size=100, alpha=0.025>
Word: cancer, Vector: [ 0.1166025  -0.00852645 -0.24493843  0.13386826 -0.0854952 ], Vector Length: 100
Word: santé, Vector: [ 0.02130495 -0.00127284 -0.05780595  0.03217423 -0.02792151], Vector Length: 100
Word: maladie, Vector: [ 0.17225628  0.00474362 -0.4092703   0.23634058 -0.12907666], Vector Length: 100
Word: traitement, Vector: [ 0.2933462  -0.00638232 -0.6850449   0.40776813 -0.21646376], Vector Length: 100
Word: virus, Vector: [ 0.09979594 -0.01248665 -0.21902901  0.11828701 -0.07449443], Vector Length: 100


Model: Word2Vec<vocab=8088, vector_size=100, alpha=0.025>
Word: cancer, Vector: [ 0.2717874   0.01005325 -0.3007892   0.19455552 -0.04255092], Vector Length: 100
Word: santé, Vector: [ 0.16421568 -0.02129341 -0.21761774  0.10137621 -0.09998267], Vector Length: 100
Word: maladie, Vector: [ 0.2939814   0.05234511 -0.29694375  0.25041983  0.04223753], Vector Length: 100
Word: traitement, Vector: [ 0.26976967  0.02016

In [11]:
# Sanity check for press models with a list of relevant words
sanityquick_check(press_models, ["politique", "économie", "sport", "culture", "technologie"])

Model: Word2Vec<vocab=38279, vector_size=100, alpha=0.025>
Word: politique, Vector: [ 1.0076609  -1.6108586   0.32492933  0.88762397  0.5507704 ], Vector Length: 100
Word: économie, Vector: [ 1.0930437   0.4297258   0.2587834   0.82179326 -0.42503133], Vector Length: 100
Word: sport, Vector: [-0.4101005   0.27533278 -0.41281456  0.09606917 -0.37837836], Vector Length: 100
Word: culture, Vector: [ 0.47177717 -0.0762701  -0.21019216  0.24507286 -0.07503431], Vector Length: 100
Word: technologie, Vector: [ 0.120883   -0.07505102 -0.03572191 -0.05439289 -0.04206685], Vector Length: 100


Model: Word2Vec<vocab=38279, vector_size=100, alpha=0.025>
Word: politique, Vector: [ 0.41200432  0.2998484  -0.02622618  0.3860958   0.12978007], Vector Length: 100
Word: économie, Vector: [ 0.13708219  0.27697608 -0.16412258  0.18987069  0.0243093 ], Vector Length: 100
Word: sport, Vector: [-0.46354666  0.07817163 -0.6657465   0.27078548 -0.11065692], Vector Length: 100
Word: culture, Vector: [ 0.2633811

## Semantic Similarity

### Comparaison of the embeddings trained on the same corpus

In [12]:
cadidat_words = ['patient', 'traitement', 'maladie', 'solution', 'jaune']

# Function to get word embeddings for a list of words from multiple models

def get_word_embeddings(models, words):
	embeddings = {}
	for model in models:
		embeddings[model] = {}
		for word in words:
			if word in model.wv:
				embeddings[model][word] = model.wv[word]
			else:
				embeddings[model][word] = None
	return embeddings



In [13]:
# Get word embeddings for the candidate words from the medical models
word_embeddings_med = get_word_embeddings(med_models, cadidat_words)

In [14]:
def compare_embeddings(embeddings):
	for model, words in embeddings.items():
		print(f"Model: {model}")
		word_embeddings = [embeddings[model][word] for word in cadidat_words]

In [15]:
# 10 Most similar words for each candidate word in the medical models

def get_most_similar_words(models, words, topn=10):
	model_results = []
 
	for model in models:
		similar_words = {}
		for word in words:
			if word in model.wv:
				similar_words[word] = model.wv.most_similar(word, topn=topn)
			else:
				similar_words[word] = None
		model_results.append(similar_words)
	return model_results

In [16]:
import pandas as pd
# CBOW model results for the candidate words in the medical models
med_model_results = get_most_similar_words(med_models, cadidat_words)

cbow_similar_words = pd.DataFrame(med_model_results[0])
cbow_similar_words

,patient,traitement,maladie,solution,jaune
0,"(par, 0.9994593262672424)","(le, 0.9996915459632874)","(en, 0.9997113347053528)","(contient, 0.999153196811676)","(ou, 0.9979560375213623)"
1,"(en, 0.9993903040885925)","(par, 0.9996468424797058)","(des, 0.999688982963562)","(flacon, 0.9991084337234497)","(avec, 0.9979474544525146)"
2,"(avec, 0.9993903040885925)","(des, 0.999641478061676)","(la, 0.9996871948242188)","((, 0.9990954995155334)","(pour, 0.997942328453064)"
3,"(pendant, 0.9993839263916016)","(sur, 0.9996342062950134)","(avec, 0.9996839165687561)","(mg, 0.999083399772644)","(ces, 0.9979398250579834)"
4,"(dans, 0.9993784427642822)","(dans, 0.9996269941329956)","(du, 0.9996662735939026)","(), 0.9990752339363098)","(résultats, 0.9979217648506165)"
5,"(le, 0.9993703961372375)","(du, 0.99961256980896)","(dans, 0.9996581077575684)","(2, 0.9990718960762024)","(:, 0.9979209303855896)"
6,"(-, 0.9993683695793152)","(en, 0.9996047019958496)","(à, 0.9996531009674072)","(,, 0.9990677237510681)","(par, 0.9979203343391418)"
7,"(après, 0.9993655681610107)","(les, 0.9995983839035034)","(au, 0.9996522665023804)","(perfusion, 0.999055802822113)","(à, 0.997917115688324)"
8,"(sous, 0.9993579387664795)","(la, 0.99959796667099)","(un, 0.9996512532234192)","(3, 0.9990448951721191)","(,, 0.9979164004325867)"
9,"(du, 0.9993451237678528)","(et, 0.9995973110198975)","(et, 0.9996500611305237)","(100, 0.9990000128746033)","(en, 0.9979045391082764)"


In [17]:
# Skipgram model results for the candidate words in the medical models
skipgram_similar_words = pd.DataFrame(med_model_results[1])
skipgram_similar_words

,patient,traitement,maladie,solution,jaune
0,"(qui, 0.9974143505096436)","(tysabri, 0.9768752455711365)","(association, 0.9904348850250244)","(flacon, 0.9806516766548157)","(13, 0.9981094002723694)"
1,"(symptômes, 0.9970325231552124)","(le, 0.9691390991210938)","(la, 0.9891632199287415)","(contient, 0.9780129194259644)","(puis, 0.9980160593986511)"
2,"(recommandé, 0.9954192042350769)","(médicaments, 0.9519137144088745)","(acide, 0.9882680773735046)","(100, 0.9777040481567383)","(24, 0.9979801177978516)"
3,"(devra, 0.9953099489212036)","(tasmar, 0.9515407681465149)","(administration, 0.9879920482635498)","(mg, 0.9773099422454834)","(boîtes, 0.9979614019393921)"
4,"(nécessaire, 0.9945858120918274)","(ayant, 0.9499891400337219)","(infection, 0.9876177310943604)","(jour, 0.9767073392868042)","(±, 0.997917652130127)"
5,"(administré, 0.9944279789924622)","(dans, 0.9499369859695435)","(prise, 0.9853360056877136)","(fois, 0.974319577217102)","(comprises, 0.9978824853897095)"
6,"(avant, 0.9939917922019958)","(diagnostic, 0.947110116481781)","(etude, 0.9841799139976501)","(300, 0.9709811806678772)","(titane, 0.9977536201477051)"
7,"(leur, 0.9930586218833923)","(epivir, 0.9469396471977234)","(cancer, 0.9840496182441711)","(injectable, 0.9656041860580444)","(diarrhée, 0.997548520565033)"
8,"(allaitement, 0.9929286241531372)","(pendant, 0.9458959698677063)","(autres, 0.983890950679779)","(9, 0.9654486775398254)","(matin, 0.9975367784500122)"
9,"(instauration, 0.9924154281616211)","(grossesse, 0.9451665282249451)","(ou, 0.9835450649261475)","(20, 0.9646579623222351)","(14, 0.9975175857543945)"


In [18]:
# FastText model results for the candidate words in the medical models
fasttext_similar_words = pd.DataFrame(med_model_results[2])
fasttext_similar_words

,patient,traitement,maladie,solution,jaune
0,"(patiente, 0.9999988079071045)","(taaitement, 0.9999986886978149)","(malade, 0.9999922513961792)","(dissolution, 0.9999971985816956)","(une, 0.9999915361404419)"
1,"(conscients, 0.9999949336051941)","(allaitement, 0.9999976754188538)","(prolactine, 0.9999920129776001)","(évolution, 0.9999969005584717)","(immune, 0.9999901056289673)"
2,"(ingrédients, 0.9999942779541016)","(évitement, 0.9999975562095642)","(sociale, 0.9999920129776001)","(evolution, 0.9999967217445374)","(minerale, 0.9999897480010986)"
3,"(présentaient, 0.9999940991401672)","(traitements, 0.9999974370002747)","(héparine, 0.9999919533729553)","(dilution, 0.9999958276748657)","(intestinale, 0.9999896287918091)"
4,"(pansements, 0.9999938607215881)","(étroitement, 0.9999973177909851)","(de, 0.9999918937683105)","(pollution, 0.9999940991401672)","(anale, 0.9999896287918091)"
5,"(parents, 0.9999938011169434)","(hautement, 0.9999967813491821)","(maladies, 0.9999917149543762)","(élocution, 0.9999907612800598)","(unilatérale, 0.99998939037323)"
6,"(fréquemment, 0.9999938011169434)","(avortement, 0.9999962449073792)","(male, 0.9999916553497314)","(distribution, 0.9999905824661255)","(generale, 0.9999892115592957)"
7,"(étirements, 0.9999936819076538)","(fortement, 0.9999960660934448)","(intraventriculaire, 0.9999915957450867)","(déglutition, 0.9999904036521912)","(portale, 0.9999890327453613)"
8,"(saignements, 0.9999934434890747)","(ajustement, 0.9999958872795105)","(professionnelle, 0.9999915361404419)","(diminution, 0.9999901652336121)","(extracapillaire, 0.9999890327453613)"
9,"(renseignements, 0.9999934434890747)","(traitment, 0.9999958276748657)","(pharmacologie, 0.9999915361404419)","(redistribution, 0.9999896883964539)","(génitale, 0.9999890327453613)"


In [19]:
press_model_results = get_most_similar_words(press_models, cadidat_words)
cbow_similar_words_press = pd.DataFrame(press_model_results[0])
cbow_similar_words_press

,patient,traitement,maladie,solution,jaune
0,"(artificiel, 0.8893436789512634)","(contenu, 0.9386574625968933)","(reprise, 0.9172935485839844)","(seule, 0.9024460911750793)","(michael, 0.9493472576141357)"
1,"(morceau, 0.88432776927948)","(trésor, 0.9339768290519714)","(somme, 0.9166337251663208)","(difficulté, 0.899651825428009)","(baba, 0.947647750377655)"
2,"(délit, 0.8747918605804443)","(collectif, 0.9334047436714172)","(douleur, 0.9073740839958191)","(querelle, 0.8965348601341248)","(peter, 0.9463744163513184)"
3,"(constituer, 0.866542637348175)","(fréjus, 0.9256477355957031)","(bataille, 0.9071843028068542)","(garantie, 0.8938069343566895)","(maréchal, 0.9456400275230408)"
4,"(éthique, 0.8659868836402893)","(nucléaire, 0.9251104593276978)","(négociation, 0.9069401025772095)","(suffisante, 0.8929925560951233)","(lars, 0.9435587525367737)"
5,"(effort, 0.8616836667060852)","(logement, 0.9243171215057373)","(procédure, 0.9038367867469788)","(forme, 0.8922521471977234)","(benhassi, 0.9426337480545044)"
6,"(mandela, 0.8604119420051575)","(stockage, 0.924147367477417)","(proposition, 0.903018057346344)","(espèce, 0.8920202255249023)","(circulant, 0.9418230652809143)"
7,"(lac, 0.8560470938682556)","(scandale, 0.9236848950386047)","(dictature, 0.9012066721916199)","(catastrophe, 0.8886871337890625)","(canadien, 0.9406613111495972)"
8,"(carnet, 0.8554087281227112)","(parc, 0.9200255870819092)","(capacité, 0.8986089825630188)","(dimension, 0.8864582180976868)","(miguel, 0.9401295185089111)"
9,"(numérique, 0.8553400635719299)","(sauvegarder, 0.9199608564376831)","(motion, 0.8975754380226135)","(bataille, 0.8858948349952698)","(oscar, 0.9399107098579407)"


In [20]:
skipgram_similar_words_press = pd.DataFrame(press_model_results[1])
skipgram_similar_words_press                                    

,patient,traitement,maladie,solution,jaune
0,"(motivation, 0.9535850286483765)","(mépris, 0.9090145826339722)","(prime, 0.8482617139816284)","(légitimité, 0.8594435453414917)","(maillot, 0.9272045493125916)"
1,"(accumulé, 0.9504270553588867)","(trésor, 0.9035193920135498)","(drogue, 0.83138108253479)","(pacifique, 0.8562818765640259)","(cooke, 0.9051808714866638)"
2,"(identification, 0.9491875767707825)","(potentiel, 0.899401843547821)","(décote, 0.8251187205314636)","(consensuelle, 0.8535175919532776)","(emparé, 0.9007589221000671)"
3,"(brevet, 0.9432902336120605)","(coût, 0.8987671732902527)","(pauvreté, 0.8246817588806152)","(difficulté, 0.8534796237945557)","(sprint, 0.8980072736740112)"
4,"(accuser, 0.9432501196861267)","(fonctionnement, 0.8925390839576721)","(nationalité, 0.8219448328018188)","(crédibilité, 0.8496129512786865)","(baden, 0.8828755617141724)"
5,"(idéologie, 0.942986011505127)","(collectif, 0.8924654722213745)","(nourriture, 0.8212154507637024)","(définition, 0.8487312197685242)","(kamch, 0.8792145252227783)"
6,"(réciprocité, 0.9427124857902527)","(financier, 0.8919105529785156)","(inflation, 0.8209903240203857)","(règle, 0.8486506938934326)","(requin, 0.8784470558166504)"
7,"(circonstance, 0.942583441734314)","(alimentaire, 0.8904927372932434)","(crainte, 0.8190812468528748)","(continuité, 0.8484598398208618)","(quadruple, 0.8773468136787415)"
8,"(irréaliste, 0.9423349499702454)","(fondamental, 0.8886478543281555)","(puissance, 0.8145581483840942)","(garantie, 0.8474835753440857)","(dae-jung, 0.8769698143005371)"
9,"(théorique, 0.9419543743133545)","(financement, 0.8875755667686462)","(nets, 0.8144540190696716)","(alternative, 0.8454848527908325)","(ilton, 0.8757629990577698)"


In [21]:
fasttext_similar_words_press = pd.DataFrame(press_model_results[2])
fasttext_similar_words_press

,patient,traitement,maladie,solution,jaune
0,"(patientent, 0.9905865788459778)","(concrètement, 0.9823235869407654)","(malnutrie, 0.9700891375541687)","(révolution, 0.9917653203010559)","(abdoune, 0.9741876125335693)"
1,"(impatient, 0.986295759677887)","(comportement, 0.9817366600036621)","(trilogie, 0.9677532911300659)","(résolution, 0.9914491176605225)","(lune, 0.9736396074295044)"
2,"(trient, 0.9857184290885925)","(farouchement, 0.9815629720687866)","(magie, 0.9573509693145752)","(évolution, 0.9888978600502014)","(brune, 0.9629074931144714)"
3,"(détient, 0.9843725562095642)","(promptement, 0.9811192154884338)","(ossétie, 0.9566081762313843)","(pollution, 0.9881457090377808)","(saadoune, 0.9592514634132385)"
4,"(coefficient, 0.9837183952331543)","(fondement, 0.9791573882102966)","(démographie, 0.9561638832092285)","(dissolution, 0.9875152111053467)","(saâdoune, 0.9581713676452637)"
5,"(ratifient, 0.983310341835022)","(vachement, 0.9788851141929626)","(cancérologie, 0.9555175304412842)","(dilution, 0.987322211265564)","(lagune, 0.9490262866020203)"
6,"(prient, 0.9832955002784729)","(faussement, 0.9781965613365173)","(colombie, 0.9529547095298767)","(caution, 0.9866690635681152)","(laâyoune, 0.9349125623703003)"
7,"(initient, 0.9817460179328918)","(dévouement, 0.9778968095779419)","(catégorie, 0.952166736125946)","(potion, 0.9798251390457153)","(dune, 0.9323375225067139)"
8,"(répandent, 0.9812501072883606)","(recrutement, 0.977849006652832)","(théologie, 0.950951337814331)","(sensation, 0.9795913696289062)","(l'une, 0.9270222187042236)"
9,"(abstient, 0.9809592962265015)","(fermement, 0.9774121642112732)","(toxicologie, 0.9504730105400085)","(démolition, 0.9787234663963318)","(aune, 0.9213128685951233)"
